# Preprocessing
We want to filter problems where distractor generation does not need the distractors to succeed (i.e., no "which of the following is correct?" problems). To this extent we let an LLM judge solvability (bc lots of cases are trivially unsolvable) but ultimately inspect the full dataset.

In [ ]:
import os
from typing import List, Dict, Tuple
import json
import re
from tqdm import tqdm

from dotenv import load_dotenv
from openai import OpenAI

import scipy.stats as st
import pandas as pd
import numpy as np

from src.datasets import EEDIDataset

from src.prompt_util import prompt_openai
from src.model_configurations import gpt_5_mini_config

load_dotenv()

### Dataset

In [ ]:
data_folder = "eedi_data"
dataset = EEDIDataset(data_folder)
print(f"We have {len(dataset)} questions")

### Prompting

In [7]:
model_config = gpt_5_mini_config
client = OpenAI(base_url=model_config["base_url"], api_key=os.environ.get(model_config["api_key_var"], None))

output_folder = os.path.join(data_folder, f"{model_config['model']}_temp_{model_config['completion_kwargs']['temperature']}_distr_annot")
os.makedirs(output_folder, exist_ok=True)

In [ ]:
def save_json(obj, path):
    with open(path, "w") as f:
        json.dump(obj, f)

def load_json(path):
    with open(path, "r") as f:
        return json.load(f)

In [ ]:
# Mark all errors (DistractorX_Error) that occur more than once within the same problem as ambiguous
error_ambiguity_by_datapointid = {}

for i in range(len(dataset)):
    problem = dataset[i]["Problem"]
    # Collect all error descriptions (keys ending with '_Error')
    error_texts = [problem[k] for k in problem if k.endswith("_Error")]
    # Count occurrences of each error string
    error_counts = {}
    for err in error_texts:
        error_counts[err] = error_counts.get(err, 0) + 1
    # Mark ambiguous errors (occur more than once)
    error_ambiguity = {}
    for k in problem:
        if k.endswith("_Error"):
            err_text = problem[k]
            error_ambiguity[k.replace("_Error", "_ambiguous")] = error_counts[err_text] > 1
    error_ambiguity_by_datapointid[str(i)] = error_ambiguity

save_json(error_ambiguity_by_datapointid, os.path.join(output_folder, "error_ambiguity_by_datapointid.json"))

In [21]:
error_ambiguity_by_datapointid = load_json(os.path.join(output_folder, "error_ambiguity_by_datapointid.json"))

In [16]:
# Ask the LLM to judge whether a problem is solvable
fewshot_solvable = (
    "Question: What is 5 + 7?\n"
    "Is this math problem solvable by a student with the information given? Answer yes or no.\n"
    "A: yes\n\n"
    "Question: What is the mode of the following numbers? \[ 1,1,4,6,7,7 \]\n"
    "Is this math problem solvable by a student with the information given? Answer yes or no.\n"
    "A: yes\n\n"
    "Question: Which of the numbers is the same when rounded to \( 2 \) decimal places or to \( 2 \) significant figures?\n"
    "Is this math problem solvable by a student with the information given? Answer yes or no.\n"
    "A: no\n\n"
)

problem_solvable_by_datapointid = {}
for i in tqdm(range(len(dataset))):
    problem = dataset[i]["Problem"]
    prompt = (
        fewshot_solvable +
        f"Question: {problem['Question']}\n"
        "Is this math problem solvable by a student with the information given? Answer yes or no."
    )
    response = prompt_openai(client, "You are a math teacher.", prompt, model_config)
    problem_solvable_by_datapointid[str(i)] = ("yes" in response.lower())
    if i % 10 == 0:
        save_json(problem_solvable_by_datapointid, os.path.join(output_folder, "problem_solvable_by_datapointid.json"))
save_json(problem_solvable_by_datapointid, os.path.join(output_folder, "problem_solvable_by_datapointid.json"))

100%|██████████| 562/562 [29:18<00:00,  3.13s/it]  


In [22]:
problem_solvable_by_datapointid = load_json(os.path.join(output_folder, "problem_solvable_by_datapointid.json"))

### Full Manual Inspection

In [ ]:
# inspect problem solvable
for dpid,dp in enumerate(dataset):
    print(dpid)
    print(dp["Problem"]["Question"])
    print(problem_solvable_by_datapointid[str(dpid)])
    print("-"*30)

In [39]:
# correction after manual inspection
for dpid in [20, 43, 49, 100, 129, 162, 247, 511]:
    problem_solvable_by_datapointid[str(dpid)] = False

for dpid in [54, 175, 183, 185, 190, 223, 254, 300, 311, 312, 349, 362, 364, 401, 408, 463, 468, 500]:
    problem_solvable_by_datapointid[str(dpid)] = True

save_json(problem_solvable_by_datapointid, os.path.join(output_folder, "problem_solvable_by_datapointid_corrected.json"))

In [41]:
from collections import Counter
Counter(problem_solvable_by_datapointid.values())

Counter({True: 483, False: 79})

# Convert From DatapointId to QuestionId

In [8]:
import json

with open("eedi_data/gpt-5-mini-2025-08-07_temp_1.0_distr_annot/problem_solvable_by_datapointid_corrected.json", "r") as f:
    data = json.load(f)

data_by_questionid = {dataset.questionids[int(k)]: v for k,v in data.items()}
with open("eedi_data/problem_solvable_by_questionid_corrected.json", "w") as f:
    json.dump(data_by_questionid, f)